In [3]:
# [Cell 1] 라이브러리 임포트 및 환경 설정 (Config)

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import math

# ==========================================
# 1. 하이퍼파라미터 및 환경 설정 (Config)
# ==========================================
class Config:
    # [핵심 변경점] 4에서 5로 변경 (Neutral 추가)
    NUM_CLASSES = 5 
    
    # 클래스 맵핑에 neutral 추가 (값은 4)
    CLASS_MAP = {'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3, 'neutral': 4}
    
    # 해상도 설정 (EfficientNet과 ViT 각각에 맞춤)
    IMG_SIZE_CNN = 260 # EfficientNet-B2
    IMG_SIZE_VIT = 224 # ViT
    
    # RTX 3070 (8GB VRAM) 환경 기준 안전한 Batch Size (OOM 발생 시 16으로 하향)
    BATCH_SIZE = 32
    EPOCHS = 30
    LEARNING_RATE = 1e-4
    
    # ArcFace 하이퍼파라미터
    ARCFACE_S = 30.0
    ARCFACE_M = 0.50
    
    # 데이터 경로 (실제 로컬 경로에 맞게 수정)
    TRAIN_DIR = "/workspace/데이터셋/img/train"
    VAL_DIR = "/workspace/데이터셋/img/val"
    
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. 시드 고정 함수 (재현성 확보)
# ==========================================
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True # 고정된 입력 크기일 경우 연산 속도 향상

seed_everything(Config.SEED)
print(f"✅ 설정 완료! Device: {Config.DEVICE}")
print(f"✅ 사용 클래스 맵핑: {Config.CLASS_MAP}")

✅ 설정 완료! Device: cuda
✅ 사용 클래스 맵핑: {'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3, 'neutral': 4}


In [4]:
# [Cell 2] 데이터 전처리(Augmentation) 및 Custom Dataset / DataLoader 구성

# ==========================================
# 1. 데이터 증강 및 전처리 (Albumentations)
# ==========================================
# EfficientNet용 (260x260)
train_transforms_cnn = A.Compose([
    A.Resize(Config.IMG_SIZE_CNN, Config.IMG_SIZE_CNN),
    A.HorizontalFlip(p=0.5),           # 50% 확률로 좌우 반전
    A.RandomBrightnessContrast(p=0.2), # 밝기/대비 랜덤 조절
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5), # 약간의 이동, 확대, 회전
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), # ImageNet 표준 정규화
    ToTensorV2()
])

val_transforms_cnn = A.Compose([
    A.Resize(Config.IMG_SIZE_CNN, Config.IMG_SIZE_CNN),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# ViT용 (224x224) - ViT는 구조 특성상 CNN보다 증강을 조금 약하게 주는 것이 좋습니다.
train_transforms_vit = A.Compose([
    A.Resize(Config.IMG_SIZE_VIT, Config.IMG_SIZE_VIT),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)), # ViT는 -1~1 사이 정규화를 선호하기도 함 (timm 기본 설정)
    ToTensorV2()
])

val_transforms_vit = A.Compose([
    A.Resize(Config.IMG_SIZE_VIT, Config.IMG_SIZE_VIT),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2()
])


# ==========================================
# 2. Custom Dataset 클래스
# ==========================================
class EmotionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # 하위 폴더명을 클래스 이름으로 읽어들임
        for class_name in os.listdir(root_dir):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
                
            # Config.CLASS_MAP에 있는 폴더만 학습에 사용 (오타나 불필요한 폴더 무시)
            lower_class_name = class_name.lower()
            if lower_class_name not in Config.CLASS_MAP:
                continue
                
            label = Config.CLASS_MAP[lower_class_name]
            
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # BGR -> RGB 변환
        
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label

# ==========================================
# 3. 데이터 로더 (DataLoader) 생성
# ==========================================
# --- EfficientNet용 ---
train_dataset_cnn = EmotionDataset(Config.TRAIN_DIR, transform=train_transforms_cnn)
val_dataset_cnn = EmotionDataset(Config.VAL_DIR, transform=val_transforms_cnn)

train_loader_cnn = DataLoader(train_dataset_cnn, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader_cnn = DataLoader(val_dataset_cnn, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# --- ViT용 ---
train_dataset_vit = EmotionDataset(Config.TRAIN_DIR, transform=train_transforms_vit)
val_dataset_vit = EmotionDataset(Config.VAL_DIR, transform=val_transforms_vit)

train_loader_vit = DataLoader(train_dataset_vit, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader_vit = DataLoader(val_dataset_vit, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"✅ CNN Dataset 구조: Train({len(train_dataset_cnn)}장), Val({len(val_dataset_cnn)}장)")
print(f"✅ ViT Dataset 구조: Train({len(train_dataset_vit)}장), Val({len(val_dataset_vit)}장)")

✅ CNN Dataset 구조: Train(5996장), Val(1200장)
✅ ViT Dataset 구조: Train(5996장), Val(1200장)


In [5]:
# [Cell 3] ArcFace Loss 정의 및 모델 구조(EfficientNet, ViT) 세팅

# ==========================================
# 1. ArcFace 마진 연산 클래스
# ==========================================
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features # = Config.NUM_CLASSES (현재 5)
        self.s = s # 스케일링 팩터 (일반적으로 30~64 사용)
        self.m = m # 각도 마진 패널티 (일반적으로 0.35~0.5 사용)
        
        # 가중치 텐서 생성 (분류기 역할)
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight) # 초기화

        # 마진 관련 사전 계산값
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, input, label):
        # 1. 입력 피처와 가중치를 각각 L2 정규화 (코사인 유사도 계산을 위해)
        cosine = nn.functional.linear(nn.functional.normalize(input), nn.functional.normalize(self.weight))
        
        # 2. 코사인 값을 다시 각도(사인)로 변환
        sine = torch.sqrt(1.0 - torch.pow(cosine, 2))
        
        # 3. 정답 클래스에만 마진(m) 추가 (cos(theta + m) = cos_theta * cos_m - sin_theta * sin_m)
        phi = cosine * self.cos_m - sine * self.sin_m
        
        # 각도가 180도(pi)를 넘어가는 경우에 대한 예외 처리 (테일러 전개 근사치)
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # 4. 정답(Label) 위치의 코사인 값만 마진이 적용된 phi로 교체하기 위한 마스킹 작업
        one_hot = torch.zeros(cosine.size(), device=Config.DEVICE)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        
        # 정답이 아닌 클래스는 기존 코사인 값 유지, 정답 클래스만 마진 적용된 값으로 대체
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        
        # 5. 스케일링(s) 곱하기
        output *= self.s
        
        return output

# ==========================================
# 2. Backbone 모델 생성 함수
# ==========================================
def build_model(model_name='efficientnet_b2', num_classes=Config.NUM_CLASSES, pretrained=True):
    print(f"[{model_name}] 모델 뼈대 생성 중... (클래스 수: {num_classes})")
    
    # 1. timm에서 사전 학습된 뼈대 로드
    model = timm.create_model(model_name, pretrained=pretrained)
    
    # 2. 모델 구조에 따라 마지막 추출(Feature) 크기를 가져옴
    if 'efficientnet' in model_name:
        in_features = model.classifier.in_features
        # 기존 분류기를 Identity(통과)로 변경 (특징만 뽑아내기 위해)
        model.classifier = nn.Identity() 
    elif 'vit' in model_name:
        in_features = model.head.in_features
        model.head = nn.Identity()
    else:
        raise ValueError("지원하지 않는 모델 구조입니다.")

    # 3. ArcFace 모듈 생성 및 모델 객체에 부착
    arcface = ArcMarginProduct(in_features, num_classes, s=Config.ARCFACE_S, m=Config.ARCFACE_M)
    
    return model, arcface

# ==========================================
# 3. 모델 초기화 및 VRAM 할당
# ==========================================
# EfficientNet-B2 생성 및 GPU 이동
model_cnn, arcface_cnn = build_model(model_name='efficientnet_b2')
model_cnn = model_cnn.to(Config.DEVICE)
arcface_cnn = arcface_cnn.to(Config.DEVICE)

# ViT Base 생성 및 GPU 이동
# vit_base_patch16_224를 사용 (기존에 85% 이상의 성능을 보여준 안정적인 모델)
model_vit, arcface_vit = build_model(model_name='vit_base_patch16_224')
model_vit = model_vit.to(Config.DEVICE)
arcface_vit = arcface_vit.to(Config.DEVICE)

print("✅ 모델 및 ArcFace 모듈 생성 및 GPU 할당 완료!")

[efficientnet_b2] 모델 뼈대 생성 중... (클래스 수: 5)


[vit_base_patch16_224] 모델 뼈대 생성 중... (클래스 수: 5)
✅ 모델 및 ArcFace 모듈 생성 및 GPU 할당 완료!


In [ ]:
# [Cell 4] 학습 파이프라인 및 루프 실행 (Early Stopping 추가)

import time
import copy

# ==========================================
# 1. 손실 함수 및 AMP 스케일러 설정
# ==========================================
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

# ==========================================
# 2. 범용 학습/검증 함수 정의 (Early Stopping 로직 포함)
# ==========================================
def train_and_evaluate(model, arcface, train_loader, val_loader, model_name, epochs=Config.EPOCHS, patience=5):
    print(f"\n{'='*50}\n🚀 [{model_name}] 학습 시작 (총 {epochs} Epochs, 인내심: {patience})\n{'='*50}")
    
    optimizer = optim.AdamW(list(model.parameters()) + list(arcface.parameters()), 
                            lr=Config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    best_arcface_wts = copy.deepcopy(arcface.state_dict())
    
    # [추가됨] 얼리 스탑 카운터
    early_stop_counter = 0 

    for epoch in range(epochs):
        start_time = time.time()
        
        # ----------------- TRAIN -----------------
        model.train()
        arcface.train()
        running_loss = 0.0
        corrects = 0
        total = 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{epochs} [Train]", leave=False)
        for inputs, labels in train_bar:
            inputs = inputs.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                features = model(inputs)
                outputs = arcface(features, labels)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)
            
            train_bar.set_postfix({'loss': f"{loss.item():.4f}"})

        train_loss = running_loss / total
        train_acc = corrects.double() / total
        scheduler.step()

        # ----------------- VALIDATION -----------------
        model.eval()
        arcface.eval()
        val_loss = 0.0
        val_corrects = 0
        val_total = 0

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{epochs} [Val]", leave=False)
            for inputs, labels in val_bar:
                inputs = inputs.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                with torch.amp.autocast('cuda'):
                    features = model(inputs)
                    outputs = arcface(features, labels)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += inputs.size(0)

        val_loss = val_loss / val_total
        val_acc = val_corrects.double() / val_total
        epoch_time = time.time() - start_time

        print(f"Epoch {epoch+1:02d}/{epochs} | Time: {epoch_time:.0f}s | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # ----------------- EARLY STOPPING CHECK -----------------
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            best_arcface_wts = copy.deepcopy(arcface.state_dict())
            print(f"  🌟 Best Val Acc 갱신! ({best_val_acc:.4f})")
            early_stop_counter = 0 # 최고 점수 갱신 시 카운터 초기화
        else:
            early_stop_counter += 1
            print(f"  ⚠️ 성능 개선 없음 (Early Stopping 스택: {early_stop_counter}/{patience})")

        # 인내심(Patience) 수치에 도달하면 강제 종료
        if early_stop_counter >= patience:
            print(f"\n🛑 조기 종료(Early Stopping) 발동! {epoch+1} Epoch에서 학습을 강제 중단합니다.")
            break

    # 학습 종료 후 최고 가중치 저장
    save_path = f"{model_name}_5class_best.pth"
    torch.save({
        'model_state_dict': best_model_wts,
        'arcface_state_dict': best_arcface_wts,
        'val_acc': best_val_acc
    }, save_path)
    
    print(f"✅ {model_name} 학습 완료! 최고 Val Acc: {best_val_acc:.4f} -> [{save_path}] 저장됨.")
    
    model.load_state_dict(best_model_wts)
    arcface.load_state_dict(best_arcface_wts)
    return model, arcface

# ==========================================
# 3. 순차적 학습 실행
# ==========================================
trained_cnn, trained_arc_cnn = train_and_evaluate(
    model=model_cnn, arcface=arcface_cnn, 
    train_loader=train_loader_cnn, val_loader=val_loader_cnn, 
    model_name="EfficientNet-B2", epochs=Config.EPOCHS
)

trained_vit, trained_arc_vit = train_and_evaluate(
    model=model_vit, arcface=arcface_vit, 
    train_loader=train_loader_vit, val_loader=val_loader_vit, 
    model_name="ViT-Base", epochs=Config.EPOCHS
)


🚀 [EfficientNet-B2] 학습 시작 (총 30 Epochs, 인내심: 5)


Epoch 01/30 [Train]:   0%|          | 0/188 [00:00<?, ?it/s]

Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 01/30 [Train]:   6%|▌         | 11/188 [00:06<01:17,  2.28it/s, loss=15.3343]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 01/30 [Train]:  11%|█         | 20/188 [00:10<01:03,  2.66it/s, loss=15.6997]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 01/30 [Train]:  11%|█         | 21/188 [00:11<01:31,  1.82it/s, loss=15.5151]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 01/30 [Train]:  13%|█▎        | 24/188 [00:12<01:00,  2.70it/s, loss=15.5280]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 01/30 [Train]:  13%|█▎        | 25/188 [00:13<01:19,  2.05it/s, loss=15.2935]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 01/30 [Train]:  15%|

Epoch 01/30 | Time: 185s | Train Loss: 11.4586 Acc: 0.1915 | Val Loss: 8.2660 Acc: 0.5208
  🌟 Best Val Acc 갱신! (0.5208)


Epoch 02/30 [Train]:   0%|          | 0/188 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 02/30 [Train]:   4%|▍         | 8/188 [00:04<01:08,  2.63it/s, loss=6.6639]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 02/30 [Train]:  13%|█▎        | 24/188 [00:12<00:53,  3.05it/s, loss=5.3988]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 02/30 [Train]:  13%|█▎        | 25/188 [00:13<01:29,  1.83it/s, loss=5.6528]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 02/30 [Train]:  16%|█▋        | 31/188 [00:15<01:04,  2.44it/s, loss=7.2232]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 02/30 [Train]:  19%|█▊        | 35/188 [00:17<00:57,  2.65it/s, loss=9.2386]Invalid SOS parameters for sequential JPEG
Invalid SOS par

Epoch 02/30 | Time: 100s | Train Loss: 5.7682 Acc: 0.6301 | Val Loss: 7.4222 Acc: 0.5758
  🌟 Best Val Acc 갱신! (0.5758)


... (3 outputs left out)

Epoch 04/30 | Time: 107s | Train Loss: 3.3952 Acc: 0.7827 | Val Loss: 7.2027 Acc: 0.6483
  🌟 Best Val Acc 갱신! (0.6483)


Epoch 05/30 [Train]:   4%|▍         | 8/188 [00:04<01:03,  2.85it/s, loss=1.8279]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 05/30 [Train]:   6%|▌         | 11/188 [00:06<01:29,  1.99it/s, loss=1.7674]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 05/30 [Train]:  10%|█         | 19/188 [00:10<01:07,  2.51it/s, loss=1.8962]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 05/30 [Train]:  13%|█▎        | 24/188 [00:12<00:56,  2.92it/s, loss=3.7568]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 05/30 [Train]:  22%|██▏       | 41/188 [00:20<01:12,  2.02it/s, loss=1.3644]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Epoch 05/30 [Train]:  23%|██▎       | 44/188 [00:21<00:46,  3.06it/s, loss=

KeyboardInterrupt: 

: 